# `mc.searoute` — Sea-distance matrix builder

This notebook builds `sea_distance_matrix.pkl`, the pre-computed all-pairs sea-distance matrix used by `markov_voyage_generator.py`.

In [ ]:
!pip install searoute tqdm

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload cluster.m1.labels.k5.csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import ast, pickle, os
import pandas as pd
from math import atan2, cos, radians, sin, sqrt
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import searoute as sr

KM_TO_NM        = 0.5399
CIRCUITY_FACTOR = 1.3
EARTH_RADIUS_KM = 6371

# Replace these placeholders with the corresponding paths in your own Google Drive.
CSV_PATH        = '<YOUR_DRIVE_PATH>/cluster.m1.labels.k5.csv'
PKL_PATH        = '<YOUR_DRIVE_PATH>/sea_distance_matrix.pkl'
CHECKPOINT_PATH = '<YOUR_DRIVE_PATH>/sea_distance_checkpoint.pkl'
SAVE_EVERY      = 5000  # Save to Drive every 5000 completed pairs.

def haversine_km(lon1, lat1, lon2, lat2):
    lat1r, lat2r = radians(lat1), radians(lat2)
    dlat = lat2r - lat1r
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(lat1r)*cos(lat2r)*sin(dlon/2)**2
    return EARTH_RADIUS_KM * 2 * atan2(sqrt(a), sqrt(1-a))

def get_sea_distance_km(lon1, lat1, lon2, lat2):
    try:
        route = sr.searoute([lon1, lat1], [lon2, lat2], units='km')
        return route.properties['length']
    except Exception:
        return None

def compute_pair(pair):
    pa, pb, lon1, lat1, lon2, lat2 = pair
    d_km = get_sea_distance_km(lon1, lat1, lon2, lat2)
    if d_km is not None:
        source = 'searoute'
    else:
        d_km   = haversine_km(lon1, lat1, lon2, lat2) * CIRCUITY_FACTOR
        source = 'fallback'
    return pa, pb, d_km, d_km * KM_TO_NM, source

# ── Extract port coordinates ──
df = pd.read_csv(CSV_PATH)
port_coords = {}
for _, row in df.iterrows():
    for p, lat, lon in zip(ast.literal_eval(row['ports']),
                           ast.literal_eval(row['latitudes']),
                           ast.literal_eval(row['longitudes'])):
        if p not in port_coords:
            port_coords[p] = (lon, lat)

ports   = list(port_coords.keys())
n_pairs = len(ports) * (len(ports) - 1) // 2
print(f"Unique ports: {len(ports)}, total port pairs: {n_pairs:,}")

# ── Load existing checkpoint (if any) ──
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'rb') as f:
        checkpoint = pickle.load(f)
    sea_dist_km = checkpoint['km']
    sea_dist_nm = checkpoint['nm']
    source_log  = checkpoint['source']
    done_pairs  = set(checkpoint['source'].keys())
    print(f"Checkpoint loaded: {len(done_pairs):,} pairs already done, resuming remainder")
else:
    sea_dist_km, sea_dist_nm, source_log = {}, {}, {}
    done_pairs  = set()
    print("No checkpoint found, starting from scratch")

# ── Build remaining task list ──
tasks = [
    (pa, pb, *port_coords[pa], *port_coords[pb])
    for pa, pb in combinations(ports, 2)
    if (pa, pb) not in done_pairs
]
print(f"Pairs remaining to compute: {len(tasks):,}")

# ── Parallel computation with periodic checkpointing ──
n_searoute = n_fallback = 0
completed  = 0

def save_checkpoint():
    with open(CHECKPOINT_PATH, 'wb') as f:
        pickle.dump(
            {'km': sea_dist_km, 'nm': sea_dist_nm, 'source': source_log},
            f, protocol=pickle.HIGHEST_PROTOCOL
        )

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(compute_pair, t): t for t in tasks}
    with tqdm(total=len(tasks), desc='Computing') as pbar:
        for future in as_completed(futures):
            pa, pb, d_km, d_nm, source = future.result()
            sea_dist_km[(pa,pb)] = sea_dist_km[(pb,pa)] = d_km
            sea_dist_nm[(pa,pb)] = sea_dist_nm[(pb,pa)] = d_nm
            source_log[(pa,pb)]  = source
            if source == 'searoute':
                n_searoute += 1
            else:
                n_fallback += 1
            completed += 1
            pbar.update(1)

            # Save to Drive every SAVE_EVERY completed pairs.
            if completed % SAVE_EVERY == 0:
                save_checkpoint()
                pbar.set_postfix({'saved': f'{completed:,}'})

# ── Save the final result once everything is done ──
with open(PKL_PATH, 'wb') as f:
    pickle.dump(
        {'km': sea_dist_km, 'nm': sea_dist_nm, 'source': source_log},
        f, protocol=pickle.HIGHEST_PROTOCOL
    )

size_mb = os.path.getsize(PKL_PATH) / 1024 / 1024
print(f"\nsearoute successes: {n_searoute:,} pairs")
print(f"Haversine fallbacks: {n_fallback:,} pairs")
print(f"Final result saved: {PKL_PATH} ({size_mb:.1f} MB)")